In [ ]:
import numpy as np
import pandas as pd
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
)
from langchain_ollama import ChatOllama

from CIT.evaluation.utils import load_jsonl, save_jsonl


In [ ]:
all_data=load_jsonl("./src/CIT/training/data/split_balance_zeros/urls_formatted/run_22.5/new_retrieved_context/all_data_transformed.jsonl")

In [ ]:
# Create a DataFrame from the loaded data
df = pd.DataFrame(all_data)


In [ ]:
#take one instance per id
df_unique = df.drop_duplicates(subset=['id'])
df_unique.reset_index(drop=True, inplace=True)

In [ ]:
#make a list of dictionaries with the required fields
data_list = []
for index, row in df_unique.iterrows():
    data_list.append(row.to_dict())


In [ ]:
save_jsonl("data/unique_74questions.jsonl", data_list)

In [ ]:
data_list_0_urls=[sample for sample in data_list if len(sample['urls'])==0]
#save the data to a jsonl file
save_jsonl("data/data_list_0_urls.jsonl", data_list_0_urls)

# define prompts with and without citing urls

In [ ]:
prompt_with= """You are an assistant working for an internal IT service of a company called Acme Corp. Always answer within the context of the company.
You work as a RAG. \n I'll give you documents about IT services.
Answer only with the information contained in the doucments I give you, detail the steps the user has to follow to solve his issue.
Do not invent information that is not in the documents so if you cannot answer the question,
say `I don't have this information`.
When you give an information, always cite from which source (title + document url) your answer comes from. You must prove your answer with the source of the relevant documents I give you.\n
The answer should remain concise.
So the goal is to provide the detailed steps to solve the issue and to cite the source of the information you provide.
Your answer must respect the following format:\n
Sources: Title(s) of the document(s) from which you took the relevant information\n
URL: url(s) of the relevant document(s)\n
To solve this issue apply the following steps:\n
1 Step 1\n
2 Step 2\n
3 Step 3\n etc.\n

\n\nNow here is the context (documents):\n{retrieved_context}\n\n\n
Remember to simply say that you don't have the information when there is no information regarding the user's question in the context. Also, citing the sources and especially the URL(s) that prove your answer is very important. Because your answer has to rely only on the context I give you.\n
The sources that you cite must be relevant to the question.
"""

prompt_without= """You are an assistant working for an internal IT service of a company called Acme Corp. Always answer within the context of the company.
You work as a RAG. \n I'll give you tutorials about IT services.
Answer only with the context I give you, detail the steps the user has to follow to solve his issue.
Do not invent information that is not in the context so if you cannot answer the question,
say `I don't have this information`.
The answer should remain concise.
So the goal is to provide the detailed steps to solve the issue.
Your answer must respect the following format:\n
To solve this issue apply the following steps:\n
1 Step 1\n
2 Step 2\n
3 Step 3\n etc.\n

\n\nNow here is the context:\n{retrieved_context}\n
"""

# Answer with the base model

In [ ]:
llm=ChatOllama(model="llama3.1:8b",temperature=0)

In [ ]:
def answer_question(question, llm, citing_sources=True):
    retrieved_context=question['retrieved_context']
    question_message=HumanMessage(content=question['question'])
    if citing_sources:
        system_message=SystemMessage(content=prompt_with.format(retrieved_context=retrieved_context))
    else:
        system_message=SystemMessage(content=prompt_without.format(retrieved_context=retrieved_context))

    messages=[system_message, question_message]

    answer=llm.invoke(messages).content
    return answer

In [ ]:
len(data_list_0_urls)

In [ ]:
sample=np.random.choice(data_list_0_urls)
# Example usage
print(f"Sample question: {sample['question']}\n")

In [ ]:
sample=np.random.choice(data_list)
# Example usage
print(f"Sample question: {sample['question']}\n")
answer_with=answer_question(sample, llm, citing_sources=True)
print(f"Answer with citing sources:\n{answer_with}\n")
answer_without=answer_question(sample, llm, citing_sources=False)
print(f"Answer without citing sources:\n{answer_without}\n")

In [ ]:
sample=data_list_0_urls[3]  # Example sample from the list with no URLs
# Example usage
print(f"Question: {sample['question']}")
answer=answer_question(sample, llm, citing_sources=True)
print(answer)